# One package's numbers

A notebook that runs like a program: it takes a package name as a
parameter and hands back a few numbers about it as named results.
Run on its own, it reports on `express`. With
[Ganymede](https://ganymede-server.com/docs/programs), a script, a Shortcut or another notebook
runs it with a different package and reads the results by name:

```sh
ganymede execute package-stats.ipynb -p pkg=hono --results
```

## Parameters

`jupyter.params` holds the run's parameters, or `{}` when there are none.
Destructuring with a default keeps the notebook runnable by hand.

In [1]:
const { pkg = "express" } = jupyter.params as { pkg?: string };
pkg

'hono'

## Fetch

The package's registry record, and last week's downloads. Neither
endpoint needs a key.

In [2]:
const record = await (await fetch(`https://registry.npmjs.org/${pkg}`)).json();
const { downloads }: { downloads: number } =
  await (await fetch(`https://api.npmjs.org/downloads/point/last-week/${pkg}`)).json();

const stable = Object.keys(record.versions).filter((v) => /^\d+\.\d+\.\d+$/.test(v));
const latest: string = record["dist-tags"].latest;
const firstRelease = record.time.created.slice(0, 10);
const lastRelease = record.time[latest].slice(0, 10);

## Results

`jupyter.result(name, value)` publishes a value under a name. Each one
shows as a `name = value` line, and whoever ran the notebook reads it
as JSON, without parsing the output.

In [3]:
jupyter.result("package", pkg);
jupyter.result("latest", latest);
jupyter.result("stable_releases", stable.length);
jupyter.result("first_release", firstRelease);
jupyter.result("weekly_downloads", downloads);

package = "hono"

latest = "4.13.18"

stable_releases = 415

first_release = "2021-12-14"

weekly_downloads = 104726612

## Summary

For a person reading the notebook, the same numbers as a sentence.

In [4]:
jupyter.md`**${pkg}** ${latest}, released ${lastRelease}: ${stable.length} stable releases since ${firstRelease},
and ${downloads.toLocaleString("en")} downloads last week.`

**hono** 4.13.18, released 2026-11-11: 415 stable releases since 2021-12-14,
and 104,726,612 downloads last week.

## Next

- [Comparing packages](https://cdp-kernel.org/docs/example-compare-packages) runs this notebook
  once per package from another notebook, through Ganymede's host API.
- [Parameters and named results](https://cdp-kernel.org/docs/jupyter-api#parameters-and-named-results)
  has the details of `jupyter.params` and `jupyter.result`.